# 1.5 章节练习

## 概述

**这一节先检查本章的关键认识，再对已经跑通的加法程序做一次小改动。**

你不需要重新编写算子。沿用1.4的代码，只改变每个任务处理的数据量，观察分块方式怎样变化，再检查加法结果。

<p class="course-note"><small>没有950设备时，可以先完成自测和预测，再对照参考答案与本页的真实运行输出。有设备时，按1.4准备环境后运行练习。</small></p>

## 1. 回顾本章的关键认识

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">小节</th>
<th style="text-align: left; vertical-align: top;">现在应能说明什么</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">1.2 TileLang初识</td>
<td style="text-align: left; vertical-align: top;">TileLang是什么、有哪些特点，以及哪些需求下可以考虑用它开发算子。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1.3 TileLang与昇腾950</td>
<td style="text-align: left; vertical-align: top;">计算与存储硬件各做什么、tile是什么，以及TileLang处在软件体系的哪一层。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1.4 环境准备与第一个算子</td>
<td style="text-align: left; vertical-align: top;">怎样运行示例，怎样区分计算描述、编译与调用，以及怎样确认输出正确。</td>
</tr>
</tbody>
</table>

### 用自己的话回答

1. PyTorch已经能调用现成算子，哪些需求下还会考虑使用TileLang自己实现算子？
2. tile、UB和Vector分别指什么？
3. `tilelang.compile(...)`与`kernel(a, b)`分别完成什么工作？
4. 程序能够编译，打印的前八项也正确，是否就能确认整个输出正确？

<details>
<summary>参考答案：检查你的理解</summary>

1. 需要实现自定义计算、组合连续步骤，或围绕常用输入形状调整实现时，可以考虑自己开发算子。TileLang采用Python语法描述设备侧的数据与计算安排。
2. **tile是数据块，UB是存储位置，Vector是计算单元。** tile是一块按一定形状组织的数据；UB用于暂存片上的数据；Vector执行向量计算。
3. `compile`将计算描述编译为可以调用的`kernel`；`kernel(a, b)`传入具体输入并执行计算，返回本次输出。
4. 不能。编译通过不代表数学结果正确，前八项也不能代表全部数据。需要将完整输出与参考结果比较。

</details>

## 2. 只改一个分块参数

### 先预测，再对照

1.4中，两个输入各有1024项，每个任务处理256项。现在保持`N = 1024`，**只把`TILE`从256改为512**，输入数据和向量加法代码都保持不变。

```python
N = 1024
TILE = 512  # 原来是256，本次只修改这一项。
```

先不要展开答案，请填写下表，再想一想：**这次修改会改变加法结果吗？为什么？**

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">配置</th>
<th style="text-align: left; vertical-align: top;">TILE</th>
<th style="text-align: left; vertical-align: top;">任务数量</th>
<th style="text-align: left; vertical-align: top;">每个任务处理的数据量</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">修改前</td>
<td style="text-align: left; vertical-align: top;">256</td>
<td style="text-align: left; vertical-align: top;">4</td>
<td style="text-align: left; vertical-align: top;">256项</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">修改后</td>
<td style="text-align: left; vertical-align: top;">512</td>
<td style="text-align: left; vertical-align: top;">____</td>
<td style="text-align: left; vertical-align: top;">____项</td>
</tr>
</tbody>
</table>

<p class="course-note"><small>提示：本例的数据能整齐分块，任务数量由<code>N // TILE</code>得到。这里沿用给定的1024项输入和256、512两种块大小。</small></p>

<details>
<summary>参考答案：分块改变了什么</summary>

修改后有**2个任务，每个任务处理512项**。两种划分都覆盖全部1024项，只是每个任务负责的数据范围不同。

<img alt="总长度保持1024项：每块256项对应4个任务，每块512项对应2个任务" src="images/01.05_tile_comparison_v2.png" style="display: block; max-width: 100%; height: auto; margin-left: 0; margin-right: auto;"/>

输入没有变，每个位置仍计算`a[i] + b[i]`，所以**完整输出应该保持不变**。这里的任务数量是程序的工作划分，不是设备的物理核数量。

</details>

## 3. 运行修改后的程序，核对结果

### 对照完整程序

下方程序与1.4相比，**只将`TILE = 256`改成了`TILE = 512`**。其余计算和校验代码完全相同，可展开对照自己的修改。

<p class="course-note"><small>课程附带的练习程序位于<code>src/01.05_grid_practice.py</code>，其中已设置<code>TILE = 512</code>。首行<code>%%writefile</code>仅用于Notebook保存文件；<code>Writing</code>或<code>Overwriting</code>表示保存完成。</small></p>

In [1]:
%%writefile src/01.05_grid_practice.py
import torch
import torch_npu
import tilelang
import tilelang.language as T

tilelang.disable_cache()  # 本例每次运行都重新编译。

N = 1024
TILE = 512


# 1. 描述设备上怎样计算。
@T.prim_func
def vector_add(
    A: T.Buffer((N,), "float32"),
    B: T.Buffer((N,), "float32"),
    C: T.Buffer((N,), "float32"),
):
    with T.Kernel(N // TILE) as bx:
        a_ub = T.alloc_shared((TILE,), "float32")
        b_ub = T.alloc_shared((TILE,), "float32")
        c_ub = T.alloc_shared((TILE,), "float32")

        # 搬入当前数据块。
        begin = bx * TILE
        T.copy(A[begin : begin + TILE], a_ub)
        T.copy(B[begin : begin + TILE], b_ub)

        # 每轮加载一组数据，做向量加法，再存回UB。
        with T.SimdVF():
            mask = T.simd.pset(32)
            for r in range(TILE // 64):
                a_vec = T.simd.vld(a_ub[r * 64])
                b_vec = T.simd.vld(b_ub[r * 64])
                c_vec = T.simd.vadd(a_vec, b_vec, mask)
                T.simd.vsts(c_ub[r * 64], c_vec, mask)

        # 写回这一块结果。
        T.copy(c_ub, C[begin : begin + TILE])


# 2. 编译计算描述，得到可以调用的kernel。
kernel = tilelang.compile(vector_add, target="ascend", out_idx=-1)

# 3. 准备输入，调用kernel，再核对结果。
a = torch.arange(N, device="npu", dtype=torch.float32)
b = torch.ones(N, device="npu", dtype=torch.float32)
out = kernel(a, b)
torch.testing.assert_close(out, a + b, rtol=0, atol=0)
print("out[:8] =", out[:8].cpu().tolist())
print("Verification passed! (已比较全部1024项输出)")


Overwriting src/01.05_grid_practice.py


### 在终端运行

在课程仓库根目录进入本章目录，再运行练习程序：

```bash
cd tutorials/tilelang_operator_development/01_overview
python src/01.05_grid_practice.py
```

<p class="course-note"><small>请先按1.4加载CANN环境。程序沿用<code>tilelang.disable_cache()</code>，每次运行都会重新编译。如果使用Notebook，先执行上方保存单元，再执行下面的可选运行单元。</small></p>

### 核对修改后的输出

下面是`TILE = 512`时的真实运行输出。先看前八项是否仍为1到8，再确认是否出现<strong><code>Verification passed!</code></strong>。

In [2]:
# 可选：在Notebook中运行同一个脚本。
import os
import re
import subprocess
import sys

env = os.environ.copy()
env["PYTHONDONTWRITEBYTECODE"] = "1"
result = subprocess.run(
    [sys.executable, "src/01.05_grid_practice.py"],
    env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
# 展示真实输出时隐去开发目录提示，其他信息与退出状态保留。
output = "".join(line for line in result.stdout.splitlines(keepends=True)
                 if "Loading tilelang libs from dev root:" not in line)
output = re.sub(r"/(?:home|root)/[^\s\"']+", "<本地路径>", output)
print(output, end="")
result.check_returncode()


[W918 19:07:19.333926948 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 19:07:29  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `vector_add` with `out_idx=-1`
2026-09-18 19:07:47  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `vector_add`
out[:8] = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
Verification passed! (已比较全部1024项输出)


前八项与1.4相同，可以帮助你直观对照预测。`Verification passed!`则说明**全部1024项输出都与PyTorch计算的`a + b`一致**，完整校验已经通过。

这次练习中，每个任务处理的数据增多了，任务数量减少了，但加法结果保持不变。你已经在一个可运行的算子上修改了分块参数，并知道如何检查修改后的结果。

接下来进入[第2章](../02_basics/02.01_chapter_intro.ipynb)，进一步理解这份程序的结构，学习怎样自己描述数据搬运和向量计算。